### 📋 Notas metodológicas importantes (Vigilancia Centinela)

A partir del **28 de marzo de 2022**, España cambió su estrategia de vigilancia (Estrategia de Vigilancia y Control tras la fase aguda de la Pandemia). Esto afecta directamente a los archivos seleccionados:

1. **Población General vs. >60 años:** - Antes de marzo 2022, se contabilizaban casos de toda la población.
   - Después de esa fecha, el foco principal de casos confirmados se desplazó a personas con factores de vulnerabilidad (mayores de 60 años).
   
2. **Unificación de Series:** - Para un análisis histórico de **gravedad** (Hosp/UCI/Defunciones), deberemos concatenar los tres archivos.
   - Para un análisis de **incidencia (casos)**, debemos ser cautelosos: a partir de julio de 2023 la serie se detiene o cambia de frecuencia.

### 🛠️ Diccionario de variables clave:
- `provincia_iso`: Código ISO 3166-2 de la provincia (ej. 'M' para Madrid, 'B' para Barcelona).
- `sexo`: H (Hombres), M (Mujeres), NC (No consta).
- `grupo_edad`: Rangos decenales (0-9, 10-19... 80+).
- `num_casos`: Casos confirmados (Solo fiables para toda la población hasta marzo 2022).
- `num_hosp` / `num_uci` / `num_def`: Indicadores de presión asistencial y mortalidad.


In [1]:
import requests
import sys
from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import urljoin

# Añado la carpeta src al sistema para poder importar el modulo config
sys.path.append(str(Path.cwd().parent))
from src.config import BASE_DIR, RAW_DATA_DIR

# Función parser para rastrear los enlaces necesarios
def scrape_data():
    base_url = "https://cnecovid.isciii.es/covid19/"
    url_datos = "https://cnecovid.isciii.es/covid19/#documentaci%C3%B3n-y-datos"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/83.0.4103.116 Safari/537.36"
    }
    datas = [
        "casos_hosp_uci_def_sexo_edad_provres.csv",
        "casos_hosp_uci_def_sexo_edad_provres_60_mas.csv",
        "hosp_uci_def_sexo_edad_provres_todas_edades.csv"
    ]
    response = requests.get(url_datos, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    links_rastreo = []
    enlaces = soup.find_all("a", href=True)
    for link in enlaces:
        href = link["href"]
        # Comprobrar si el enlace coincide con alguno de nuestros datos
        if any(data in href for data in datas):
            # Si es así, convertir en URL absoluta
            full_url = urljoin(base_url, href)
            if full_url not in links_rastreo:
                links_rastreo.append(full_url)
    return links_rastreo

# Funcion para descarga de datos
def download_data(data_links):
    if not data_links:
        print("⚠️ No se encontraron datos en la página.")
        return
    for link in data_links:
        file_name = link.split("/")[-1]
        print(f"Descargando: {file_name}...")
        try:
            response= requests.get(link, timeout=10)
            response.raise_for_status()
            with open(RAW_DATA_DIR / file_name, "wb") as f:
                f.write(response.content)
            print(f"✅ Guardado en: {RAW_DATA_DIR / file_name}")
        except Exception as e:
            print(f"⚠️ Error al descargar {file_name}: {e}")          

# Ejecutar
if __name__ == "__main__":
    links = scrape_data()
    print(f"✅ Se encontraron {len(links)} enlaces en la página.")
    download_data(links)

Proyecto configurado en: /Users/filipihenrique/Desktop/covid_spain_project
✅ Se encontraron 3 enlaces en la página.
Descargando: casos_hosp_uci_def_sexo_edad_provres.csv...
✅ Guardado en: /Users/filipihenrique/Desktop/covid_spain_project/data/raw/casos_hosp_uci_def_sexo_edad_provres.csv
Descargando: casos_hosp_uci_def_sexo_edad_provres_60_mas.csv...
✅ Guardado en: /Users/filipihenrique/Desktop/covid_spain_project/data/raw/casos_hosp_uci_def_sexo_edad_provres_60_mas.csv
Descargando: hosp_uci_def_sexo_edad_provres_todas_edades.csv...
✅ Guardado en: /Users/filipihenrique/Desktop/covid_spain_project/data/raw/hosp_uci_def_sexo_edad_provres_todas_edades.csv
